# <font color = navy > <center> **Modeling and Forecasting Climate Disasters in France** </center> </font>
## <font color = navy > <center> **Part 1: Big Data - Collecting Data** </center> </font>


###  <font color = navy > <center> **How can climate disasters be anticipated using measurable environmental indicators in France?** </center> </font>

<center> <img src="presentation.jpg" style="width:800px; height:600px"> </center>
<p></p>
<center> <img src="AMU_logo.jpg" style="width:250px; height:100px"> </center>

<font color = navy > Introduce by a student from Aix Marseille University : **HUANG Eric** <font color = navy >

<font color = navy > <div align="right"> <b> 2024 - 2025 </b> </div> <font color = navy >

In [1]:
#pip install requests beautifulsoup4
# in Anaconda Prompt
    # pip install selenium 
    # pip install ChromeDriver

In [2]:
# API & Web Scrapping
import requests, os, re
from datetime import time
import datetime
import time
from concurrent.futures import ThreadPoolExecutor
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

# basic
import pandas as pd
import numpy as np

# fast
import polars as pl
import glob

# Introduction & Context
## Problem statement and objectives
- Try to predict natural disaster (pollution, fire, drought)
  - extreme pollution -> smog or fire
  - drought + high temperature (high tropospheric ozone, low NDVI) -> fire in forest
  - rain + saturated soil -> flood
## Dataset description and source
## Expected impact/applications
- Main Natural Disaster
  - June 2022: fire in forest -> 	Gironde (Landes, La Teste) & Var
  - October 2023: flood -> Pas-de-Calais & Nord
  - December 2023 - January 2024 -> Flooding in Northern France (Nord and Pas-de-Calais,  Rivers such as the Aa, Lys, Helpe Mineure, and Solre overflowed
    - over 2,000 homes were damaged, and 10,000 households experienced power outages. Additionally, 2,100 people lacked access to drinking water, and 189 municipalities were affected.
  - October 2024: Storm Kirk -> record rainfall in Île-de-France (Paris-Montsouris, Seine-et-Marne)
  - December 2024: Storm Enol -> northern France and heavy snowfall in the Alps and Pyrenees
  - January 2025: Storm Floriane -> Brittany and the Île-de-France region
  - March 2024: Flooding -> in the Southeast (Gard, Ardèche, and Var)

# Data Acquisition & Preprocessing
- data from [Data Europa](https://data.europa.eu/) -> redirect to [Data Gouv](https://www.data.gouv.fr/fr/)
  - [Polluants atmosphériques réglementés](https://files.data.gouv.fr/lcsqa/concentrations-de-polluants-atmospheriques-reglementes/temps-reel/)

## Justify your preprocessing choices

## Step 1 - Collecting pollution data

### Pollutant index

In [ ]:
# DONE (not need to run anymore), (2021, 202, 202, 202, 202)
# Web scrapping with BeautifulSoup

base_url = "https://files.data.gouv.fr/lcsqa/concentrations-de-polluants-atmospheriques-reglementes/temps-reel/2021/"  # Base URL ****** --------> change here
save_dir = "pollution_data_2022" # Create a folder to save the files ****** --------> change here 
os.makedirs(save_dir, exist_ok=True) 

response = requests.get(base_url) # Get HTML content
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')


file_extensions = ".csv"  # Find all links ending in .zip or .csv or other desired extensions
links = soup.find_all('a', href=True)

for link in links: # Filter and download files
    file_href = link['href']
    if any(file_href.endswith(ext) for ext in file_extensions):
        file_url = urljoin(base_url, file_href)
        filename = os.path.join(save_dir, os.path.basename(file_href))

        print(f"Downloading {file_url} ...")
        with requests.get(file_url, stream=True) as r:
            r.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print(f"Saved to {filename}")

print("All files downloaded")

Saved to pollution_data_2022\FR_E2_2021-01-01.csv
Saved to pollution_data_2022\FR_E2_2021-01-02.csv
Saved to pollution_data_2022\FR_E2_2021-01-03.csv
Saved to pollution_data_2022\FR_E2_2021-01-04.csv
Saved to pollution_data_2022\FR_E2_2021-01-05.csv
Saved to pollution_data_2022\FR_E2_2021-01-06.csv
Saved to pollution_data_2022\FR_E2_2021-01-07.csv
Saved to pollution_data_2022\FR_E2_2021-01-08.csv
Saved to pollution_data_2022\FR_E2_2021-01-09.csv
Saved to pollution_data_2022\FR_E2_2021-01-10.csv
Saved to pollution_data_2022\FR_E2_2021-01-11.csv
Saved to pollution_data_2022\FR_E2_2021-01-12.csv
Saved to pollution_data_2022\FR_E2_2021-01-13.csv
Saved to pollution_data_2022\FR_E2_2021-01-14.csv
Saved to pollution_data_2022\FR_E2_2021-01-15.csv
Saved to pollution_data_2022\FR_E2_2021-01-16.csv
Saved to pollution_data_2022\FR_E2_2021-01-17.csv
Saved to pollution_data_2022\FR_E2_2021-01-18.csv
Saved to pollution_data_2022\FR_E2_2021-01-19.csv
Saved to pollution_data_2022\FR_E2_2021-01-20.csv


In [6]:
# DONE (not need to run anymore), (2021, 2022, 2023, 2024, 2025)
# Merge all the dataset from the same year

# Path to your folder with all the CSV files
folder_path = "pollution_data_2022"

# Get all CSV file paths in that folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Combine all CSVs into one DataFrame
df_list = [pd.read_csv(file, low_memory=False) for file in csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

# Save to a single CSV if desired
combined_df.to_csv("combined_pollution_2022.csv", index=False)

print(f"Combined {len(csv_files)} files into one DataFrame with {combined_df.shape[0]} rows")

Combined 730 files into one DataFrame with 36150665 rows


In [ ]:
# merge all the year

folder_path = os.path.expanduser("~/Desktop/Data sciences")

# Fichiers spécifiques
years = range(2021, 2026)
csv_files = [os.path.join(folder_path, f"pollution_{year}.csv") for year in years]

# Lecture + fusion
df_list = [pd.read_csv(f, sep=",", low_memory=False, encoding="utf-8") for f in csv_files]
pollution = pd.concat(df_list, ignore_index=True)

print("Merging Success :", pollution.shape)

In [ ]:
# Export
pollution.to_csv("pollution.csv", index=False)

### Weather index

In [ ]:
# DONE (not need to run anymore)
# Web Scrapping the file's name
# URL
resources_url = "https://www.data.gouv.fr/api/2/datasets/6569b51ae64326786e4e8e1a/resources/?page=1&page_size=1000"

# Requête GET
response = requests.get(resources_url)
response.raise_for_status()
data = response.json()

# Accès aux ressources
resources = data.get('data', [])

# Extraire et afficher tous les titres de fichiers .csv.gz
print("files name .csv.gz:\n")
for res in resources:
    if isinstance(res, dict) and res.get('url', '').endswith('.csv.gz'):
        print(res.get('title', 'Titre inconnu'))

In [ ]:
# DONE (not need to run anymore)
# Web Scrapping - File of .csv.gz with 'RR-T-Vent'

resources_url = "https://www.data.gouv.fr/api/2/datasets/6569b51ae64326786e4e8e1a/resources/?page=1&page_size=1000"  # Base URL
save_dir = "meteo_data_csv" # Save in the doc
os.makedirs(save_dir, exist_ok=True)

response = requests.get(resources_url)
response.raise_for_status()
data = response.json()

resources = data.get('data', [])


matching_resources = [ # Filter
    r for r in resources
    if isinstance(r, dict) 
    and r.get('url', '').endswith('.csv.gz') # File of .csv.gz
    and 'RR-T-Vent' in r.get('title', '') # with 'RR-T-Vent'
]

print(f"🔍 {len(matching_resources)} download .csv.gz file with 'RR-T-Vent'.")

for res in tqdm(matching_resources, desc="Dowloading file"):
    url = res['url'] # URL
    title = res.get('title', '') # Title of the file
    filename = os.path.join(save_dir, os.path.basename(url)) #

    try:
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        tqdm.write(f"Saved : {filename}")
    except Exception as e:
        tqdm.write(f"Error pour {title} : {e}")

print("Files downloaded")

In [70]:
# DONE (not need to run anymore)

# fast save in the folder meteo_data_csv
# filtering all the data of .csv.gz , after 2020/12/31) #because I only from this periods

folder = "meteo_data_csv"
filtered_files = []

# Step 1: filter by .csv.gz, name, departement
for fname in os.listdir(folder):
    if not fname.endswith(".csv.gz"):
        continue
    if not ("RR-T-Vent" in fname or "autres-parametres" in fname):
        continue

    # Extract & filter the good departement from Q_XX_
    match = re.search(r"Q_(\d+)_", fname)
    if match:
        dept_num = int(match.group(1))
        if 1 <= dept_num <= 95:
            filtered_files.append(os.path.join(folder, fname))

print(f"{len(filtered_files)} files")

# Step 2: read, filter after 2020/12/31 & save
for fpath in filtered_files:
    print(f"🔄 Traitement : {os.path.basename(fpath)}")

    try:
        df = pl.read_csv(fpath, separator=";", infer_schema_length=1000)

        # 🧹 Put None to the date after 2021/12/31
        df = df.with_columns(
            pl.when(pl.col("AAAAMMJJ") > 20201231)
            .then(None)
            .otherwise(pl.col("AAAAMMJJ"))
            .alias("AAAAMMJJ")
        )

        # save
        outpath = fpath.replace(".csv.gz", "_filtered.csv")
        df.write_csv(outpath)
        print(f"File filtred & saved : {os.path.basename(outpath)}")

    except Exception as e:
        print(f"Error for {fpath} : {e}")

285 files
🔄 Traitement : Q_01_1852-1949_RR-T-Vent.csv.gz
File filtred & saved : Q_01_1852-1949_RR-T-Vent_filtered.csv
🔄 Traitement : Q_01_latest-2024-2025_RR-T-Vent.csv.gz
File filtred & saved : Q_01_latest-2024-2025_RR-T-Vent_filtered.csv
🔄 Traitement : Q_01_previous-1950-2023_RR-T-Vent.csv.gz
File filtred & saved : Q_01_previous-1950-2023_RR-T-Vent_filtered.csv
🔄 Traitement : Q_02_1864-1949_RR-T-Vent.csv.gz
File filtred & saved : Q_02_1864-1949_RR-T-Vent_filtered.csv
🔄 Traitement : Q_02_latest-2024-2025_RR-T-Vent.csv.gz
File filtred & saved : Q_02_latest-2024-2025_RR-T-Vent_filtered.csv
🔄 Traitement : Q_02_previous-1950-2023_RR-T-Vent.csv.gz
File filtred & saved : Q_02_previous-1950-2023_RR-T-Vent_filtered.csv
🔄 Traitement : Q_03_1871-1949_RR-T-Vent.csv.gz
File filtred & saved : Q_03_1871-1949_RR-T-Vent_filtered.csv
🔄 Traitement : Q_03_latest-2024-2025_RR-T-Vent.csv.gz
File filtred & saved : Q_03_latest-2024-2025_RR-T-Vent_filtered.csv
🔄 Traitement : Q_03_previous-1950-2023_RR-T-Vent

## Step 2 - Data cleaning and transformation

### Pollutant index

In [3]:
# Pandas
start = time.time()

# classic version pandas + glob & merging
folder_path = "pollution_data_2025"
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
df_list = [pd.read_csv(f, sep=";", low_memory=False, encoding='utf-8') for f in csv_files]
pollution_pd = pd.concat(df_list, ignore_index=True)

end = time.time()
print("normal version - pandas: ", round(end - start, 2), "secondes")

normal version - pandas:  41.68 secondes


In [4]:
pollution_pd

,Date de début,Date de fin,Organisme,code zas,Zas,code site,nom site,type d'implantation,Polluant,type d'influence,...,type de valeur,valeur,valeur brute,unité de mesure,taux de saisie,couverture temporelle,couverture de données,code qualité,validité,Erreur d'écriture
0,2023/01/01 00:00:00,2023/01/01 01:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO,Fond,...,moyenne horaire validée,0.7,0.725,µg-m3,NaN,NaN,NaN,A,1.0,NaN
1,2023/01/01 01:00:00,2023/01/01 02:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO,Fond,...,moyenne horaire validée,0.7,0.700,µg-m3,NaN,NaN,NaN,A,1.0,NaN
2,2023/01/01 02:00:00,2023/01/01 03:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO,Fond,...,moyenne horaire validée,0.6,0.575,µg-m3,NaN,NaN,NaN,A,1.0,NaN
3,2023/01/01 03:00:00,2023/01/01 04:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO,Fond,...,moyenne horaire validée,0.8,0.800,µg-m3,NaN,NaN,NaN,A,1.0,NaN
4,2023/01/01 04:00:00,2023/01/01 05:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO,Fond,...,moyenne horaire validée,0.7,0.725,µg-m3,NaN,NaN,NaN,A,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25268276,2025/05/15 10:00:00,2025/05/15 11:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,9.6,9.625,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25268277,2025/05/15 11:00:00,2025/05/15 12:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,7.0,7.025,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25268278,2025/05/15 12:00:00,2025/05/15 13:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,7.0,7.000,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25268279,2025/05/15 13:00:00,2025/05/15 14:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,4.2,4.225,µg-m3,NaN,NaN,NaN,A,1.0,NaN


In [5]:
pollution_pd.isna().sum()

Date de début                   0
Date de fin                     0
Organisme                       0
code zas                        0
Zas                             0
code site                       0
nom site                        0
type d'implantation             0
Polluant                        0
type d'influence                0
discriminant              3430769
Réglementaire                   0
type d'évaluation               0
procédure de mesure             0
type de valeur                  0
valeur                    1007870
valeur brute              1007870
unité de mesure                 0
taux de saisie           25268281
couverture temporelle    25268281
couverture de données    25268281
code qualité                    0
validité                        0
Erreur d'écriture        25268281
dtype: int64

In [6]:
pollution_pd['discriminant'].unique()

array(['A', 'B', '1', 'C', '2', nan, 'I', 'F', 'E', '5', '3', '4', '0',
       'D', '7', 'H', 'G', '9', 'k', 'j', 'i', 't', '6', 'Z', '8', 'O',
       'K', 'L'], dtype=object)

In [7]:
pollution_pd['taux de saisie'].unique()

array([nan])

In [8]:
pollution_pd['couverture temporelle'].unique()

array([nan])

In [9]:
pollution_pd['couverture de données'].unique()

array([nan])

In [10]:
pollution_pd["Erreur d'écriture"].unique()

array([nan], dtype=object)

In [11]:
pollution_pd['Polluant'].unique()

array(['NO', 'NO2', 'O3', 'NOX as NO2', 'PM10', 'PM2.5', 'C6H6', 'SO2',
       'CO'], dtype=object)

In [12]:
# we only need 
pollution_pd = pollution_pd[pollution_pd['Polluant'] != 'NO']
pollution_pd = pollution_pd[pollution_pd['Polluant'] != 'NOX as NO2']
pollution_pd = pollution_pd[pollution_pd['Polluant'] != 'SO2']
pollution_pd = pollution_pd[pollution_pd['Polluant'] != 'C6H6']
pollution_pd = pollution_pd[pollution_pd['Polluant'] != 'CO']

In [13]:
pollution_pd

,Date de début,Date de fin,Organisme,code zas,Zas,code site,nom site,type d'implantation,Polluant,type d'influence,...,type de valeur,valeur,valeur brute,unité de mesure,taux de saisie,couverture temporelle,couverture de données,code qualité,validité,Erreur d'écriture
24,2023/01/01 00:00:00,2023/01/01 01:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,moyenne horaire validée,5.8,5.750,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25,2023/01/01 01:00:00,2023/01/01 02:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,moyenne horaire validée,6.1,6.125,µg-m3,NaN,NaN,NaN,A,1.0,NaN
26,2023/01/01 02:00:00,2023/01/01 03:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,moyenne horaire validée,4.9,4.925,µg-m3,NaN,NaN,NaN,A,1.0,NaN
27,2023/01/01 03:00:00,2023/01/01 04:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,moyenne horaire validée,5.0,4.950,µg-m3,NaN,NaN,NaN,A,1.0,NaN
28,2023/01/01 04:00:00,2023/01/01 05:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,moyenne horaire validée,4.6,4.600,µg-m3,NaN,NaN,NaN,A,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25268276,2025/05/15 10:00:00,2025/05/15 11:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,9.6,9.625,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25268277,2025/05/15 11:00:00,2025/05/15 12:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,7.0,7.025,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25268278,2025/05/15 12:00:00,2025/05/15 13:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,7.0,7.000,µg-m3,NaN,NaN,NaN,A,1.0,NaN
25268279,2025/05/15 13:00:00,2025/05/15 14:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,FR27ZRE01,ZR BOURGOGNE-FRANCHE-COMTE,FR82070,Baume-les-Dames,Urbaine,PM2.5,Trafic,...,moyenne horaire brute,4.2,4.225,µg-m3,NaN,NaN,NaN,A,1.0,NaN


In [14]:
pollution_pd = pollution_pd.drop(columns='taux de saisie') # empty, so we can drop
pollution_pd = pollution_pd.drop(columns='couverture temporelle') # empty, so we can drop
pollution_pd = pollution_pd.drop(columns="couverture de données") # empty, so we can drop
pollution_pd = pollution_pd.drop(columns="Erreur d'écriture") # empty, so we can drop

In [15]:
# I transform the date in datetime, because I need to do the average, such as I only keep the rate at the daily rates
pollution_pd['Date de début'] = pd.to_datetime(pollution_pd['Date de début'])

In [16]:
# I spilt date & hours, so I group by date & than do the average of all the hours in that date
pollution_pd['date'] = pollution_pd['Date de début'].dt.date
pollution_pd['hours'] = pollution_pd['Date de début'].dt.time

In [17]:
pollution_pd.head()

,Date de début,Date de fin,Organisme,code zas,Zas,code site,nom site,type d'implantation,Polluant,type d'influence,...,type d'évaluation,procédure de mesure,type de valeur,valeur,valeur brute,unité de mesure,code qualité,validité,date,hours
24,2023-01-01 00:00:00,2023/01/01 01:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,mesures fixes,Auto NO2_NOx Conf meth CHIMILU,moyenne horaire validée,5.8,5.750,µg-m3,A,1.0,2023-01-01,00:00:00
25,2023-01-01 01:00:00,2023/01/01 02:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,mesures fixes,Auto NO2_NOx Conf meth CHIMILU,moyenne horaire validée,6.1,6.125,µg-m3,A,1.0,2023-01-01,01:00:00
26,2023-01-01 02:00:00,2023/01/01 03:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,mesures fixes,Auto NO2_NOx Conf meth CHIMILU,moyenne horaire validée,4.9,4.925,µg-m3,A,1.0,2023-01-01,02:00:00
27,2023-01-01 03:00:00,2023/01/01 04:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,mesures fixes,Auto NO2_NOx Conf meth CHIMILU,moyenne horaire validée,5.0,4.950,µg-m3,A,1.0,2023-01-01,03:00:00
28,2023-01-01 04:00:00,2023/01/01 05:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01011,Metz-Centre,Urbaine,NO2,Fond,...,mesures fixes,Auto NO2_NOx Conf meth CHIMILU,moyenne horaire validée,4.6,4.600,µg-m3,A,1.0,2023-01-01,04:00:00


In [18]:
# group by date & than do the average of all the hours in that date and put a dataframe
valeur = pollution_pd.groupby(['date', 'Polluant', "nom site"])['valeur'].mean().reset_index(name='valeur')
valeur_brute = pollution_pd.groupby(['date', 'Polluant', "nom site"])['valeur brute'].mean().reset_index(name='valeur brute')

In [19]:
pollution_pd.dtypes

Date de début          datetime64[ns]
Date de fin                    object
Organisme                      object
code zas                       object
Zas                            object
code site                      object
nom site                       object
type d'implantation            object
Polluant                       object
type d'influence               object
discriminant                   object
Réglementaire                  object
type d'évaluation              object
procédure de mesure            object
type de valeur                 object
valeur                        float64
valeur brute                  float64
unité de mesure                object
code qualité                   object
validité                      float64
date                           object
hours                          object
dtype: object

In [20]:
# we still have the discriminant, the valeur & the valeur brute which have NA value
pollution_pd.isna().sum()

Date de début                0
Date de fin                  0
Organisme                    0
code zas                     0
Zas                          0
code site                    0
nom site                     0
type d'implantation          0
Polluant                     0
type d'influence             0
discriminant           1908029
Réglementaire                0
type d'évaluation            0
procédure de mesure          0
type de valeur               0
valeur                  608639
valeur brute            608639
unité de mesure              0
code qualité                 0
validité                     0
date                         0
hours                        0
dtype: int64

In [21]:
# Let's check where it is, it is the case when the validité is = -1.0
pollution_pd[pollution_pd["valeur"].isna()].iloc[:,5:]

,code site,nom site,type d'implantation,Polluant,type d'influence,discriminant,Réglementaire,type d'évaluation,procédure de mesure,type de valeur,valeur,valeur brute,unité de mesure,code qualité,validité,date,hours
260,FR01014,Pont-à-Mousson,Urbaine,PM10,Fond,I,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire validée,NaN,NaN,µg-m3,N,-1.0,2023-01-01,20:00:00
2976,FR03068,TOULON FOCH,Urbaine,PM10,Trafic,1,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire validée,NaN,NaN,µg-m3,N,-1.0,2023-01-01,00:00:00
2977,FR03068,TOULON FOCH,Urbaine,PM10,Trafic,1,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire validée,NaN,NaN,µg-m3,N,-1.0,2023-01-01,01:00:00
2978,FR03068,TOULON FOCH,Urbaine,PM10,Trafic,1,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire validée,NaN,NaN,µg-m3,N,-1.0,2023-01-01,02:00:00
2979,FR03068,TOULON FOCH,Urbaine,PM10,Trafic,1,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire validée,NaN,NaN,µg-m3,N,-1.0,2023-01-01,03:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25268100,FR82043,Chatenois,Périurbaine,PM10,Industrielle,3,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire brute,NaN,NaN,µg-m3,N,-1.0,2025-05-15,14:00:00
25268190,FR82060,Vesoul Pres Caillet,Urbaine,PM10,Fond,2,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire brute,NaN,NaN,µg-m3,N,-1.0,2025-05-15,14:00:00
25268205,FR82060,Vesoul Pres Caillet,Urbaine,PM2.5,Fond,2,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire brute,NaN,NaN,µg-m3,N,-1.0,2025-05-15,14:00:00
25268265,FR82070,Baume-les-Dames,Urbaine,PM10,Trafic,1,Oui,mesures fixes,Auto PM_Conf_app BAM 1020-Smart Heater,moyenne horaire brute,NaN,NaN,µg-m3,N,-1.0,2025-05-15,14:00:00


In [22]:
# Let's check the type of quality 
pollution_pd['code qualité'].unique()
# A -> Automatique (automatiquely record by a machine)
# N -> Not good
# R -> Revisited manuelly (valided by an expert)

array(['A', 'N', 'R'], dtype=object)

In [23]:
pollution_pd['validité'].unique()

array([ 1., -1.])

In [24]:
# let's regroup the code qualité & validité to check the share of each inside of the other
pollution_pd.groupby(['code qualité', 'validité']).size().reset_index(name='count')
# We have code qualité = validité in term of number, so it is the same

,code qualité,validité,count
0,A,1.0,13791666
1,N,-1.0,608639
2,R,1.0,681735


In [25]:
# we only keep the one that is valid
pollution_pd = pollution_pd[pollution_pd["code qualité"]!='N']

In [26]:
# we check that we didn't do a mistake
pollution_pd['validité'].unique()

array([1.])

In [27]:
# verify the type de valeur
pollution_pd['type de valeur'].unique()

array(['moyenne horaire validée', 'moyenne horaire brute'], dtype=object)

In [28]:
# check the share, we have way more horaire validée than brute
pollution_pd.groupby(["type de valeur"]).size().reset_index(name='total')
# we only have 2 types, we could drop it, we don't need that, we will assume it is same type to keep simple

,type de valeur,total
0,moyenne horaire brute,98315
1,moyenne horaire validée,14375086


In [29]:
# check the content in type d'influence
pollution_pd["type d'influence"].unique()

array(['Fond', 'Industrielle', 'Trafic'], dtype=object)

In [30]:
# share of the type d'influence
pollution_pd.groupby(["type d'influence"]).size().reset_index(name='total')
# we may need that

,type d'influence,total
0,Fond,10967482
1,Industrielle,901075
2,Trafic,2604844


In [31]:
# check the content inside type d'implantation, we have 5 types
pollution_pd["type d'implantation"].unique()

array(['Urbaine', 'Rurale près des villes', 'Périurbaine',
       'Rurale régionale', 'Rurale nationale'], dtype=object)

In [32]:
# share of type d'implantation
pollution_pd.groupby(["type d'implantation"]).size().reset_index(name='total')
# we could probably need that

,type d'implantation,total
0,Périurbaine,3053328
1,Rurale nationale,433957
2,Rurale près des villes,538274
3,Rurale régionale,810279
4,Urbaine,9637563


In [33]:
# check the content inside type d'évaluation, we have 3
pollution_pd["type d'évaluation"].unique()

array(['mesures fixes', 'mesures indicatives', 'estimation objective'],
      dtype=object)

In [34]:
# share of type d'évaluation
pollution_pd.groupby(["type d'évaluation"]).size().reset_index(name='total')
# we will assume that all is good enough to keep it simple, so we could drop it

,type d'évaluation,total
0,estimation objective,96157
1,mesures fixes,13088254
2,mesures indicatives,1288990


In [35]:
# content inside Réglementaire
pollution_pd['Réglementaire'].unique()
# only one result which is reglemented, so we can drop it.

array(['Oui'], dtype=object)

In [36]:
# content inside unité de mesure
pollution_pd['unité de mesure'].unique()
# only one which the same, we can drop it, and keep that in minds

array(['µg-m3', 'µg/m3'], dtype=object)

In [37]:
# content inside procédure de mesure
pollution_pd['procédure de mesure'].unique()
# we don't need that, we will assume that the data is good enough, so we drop it.

array(['Auto NO2_NOx Conf meth CHIMILU', 'Auto O3 Conf meth PHOTO UV',
       'Auto PM_Conf_app TEOM 1405-F',
       'Auto PM_Conf_app BAM 1020-Smart Heater',
       'Auto PM_Conf_app TEOM-FDMS 8500bc', 'Auto O3 Conf app API 400E',
       'Auto O3 Conf app APOA370', 'Auto NO2_NOx Conf app API 200E',
       'Auto PM_Conf_app FIDAS 200E', 'Auto NO2_NOx app AC32e',
       'Auto PM_Conf_app FIDAS 200', 'Auto O3 Conf app Serinus10',
       'Auto NO2 NOX Conf  app APNA370', 'Auto PM_Conf_meth FDMS',
       'Auto NO2_NOx app AC32M', 'Auto O3 Conf app API T400',
       'Auto NO2_NOx app 42i', 'Auto O3 Conf app O342M',
       'Auto O3_Conf_app 49i', 'Auto O3 Conf app O342E',
       'Auto PM_Conf_app MP101M-RST', 'Auto PM_Conf_app MP101M-QAL1',
       'Auto PM_Conf_app TEOM1400AB-FDMS', 'Auto NO2_Conf_app AS32M',
       'Auto NO2_NOx Conf app API T200',
       'Auto NO2_NOx Conf app API 200UP ', 'Auto NO Conf app APNA370',
       'Auto NO2_NOx app 42i-TL', 'Auto PM_Conf_app TEOM 1405-DF',
      

In [38]:
# check the number of technique
pollution_pd['procédure de mesure'].nunique()

31

In [39]:
pollution_pd = pollution_pd.drop(columns="validité") # after filter all the observation is valided
pollution_pd = pollution_pd.drop(columns="code qualité") # after filter all the observation is of good quality
pollution_pd = pollution_pd.drop(columns="Réglementaire") # the same value, true for all; it useless 
# pollution_pd = pollution_pd.drop(columns="type d'influence") # may need
pollution_pd = pollution_pd.drop(columns="unité de mesure") # not need, keep in mind that the united is in µg/m3
pollution_pd = pollution_pd.drop(columns="code site") # duplication of site & not more information
pollution_pd = pollution_pd.drop(columns="code zas") # duplication of zas & not more information
pollution_pd = pollution_pd.drop(columns="procédure de mesure") # we don't need that
pollution_pd = pollution_pd.drop(columns="type de valeur") # both is valide, we drop it

In [40]:
pollution_pd

,Date de début,Date de fin,Organisme,Zas,nom site,type d'implantation,Polluant,type d'influence,discriminant,type d'évaluation,valeur,valeur brute,date,hours
24,2023-01-01 00:00:00,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz-Centre,Urbaine,NO2,Fond,B,mesures fixes,5.8,5.750,2023-01-01,00:00:00
25,2023-01-01 01:00:00,2023/01/01 02:00:00,ATMO GRAND EST,ZAG METZ,Metz-Centre,Urbaine,NO2,Fond,B,mesures fixes,6.1,6.125,2023-01-01,01:00:00
26,2023-01-01 02:00:00,2023/01/01 03:00:00,ATMO GRAND EST,ZAG METZ,Metz-Centre,Urbaine,NO2,Fond,B,mesures fixes,4.9,4.925,2023-01-01,02:00:00
27,2023-01-01 03:00:00,2023/01/01 04:00:00,ATMO GRAND EST,ZAG METZ,Metz-Centre,Urbaine,NO2,Fond,B,mesures fixes,5.0,4.950,2023-01-01,03:00:00
28,2023-01-01 04:00:00,2023/01/01 05:00:00,ATMO GRAND EST,ZAG METZ,Metz-Centre,Urbaine,NO2,Fond,B,mesures fixes,4.6,4.600,2023-01-01,04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25268275,2025-05-15 09:00:00,2025/05/15 10:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume-les-Dames,Urbaine,PM2.5,Trafic,1,mesures fixes,13.1,13.050,2025-05-15,09:00:00
25268276,2025-05-15 10:00:00,2025/05/15 11:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume-les-Dames,Urbaine,PM2.5,Trafic,1,mesures fixes,9.6,9.625,2025-05-15,10:00:00
25268277,2025-05-15 11:00:00,2025/05/15 12:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume-les-Dames,Urbaine,PM2.5,Trafic,1,mesures fixes,7.0,7.025,2025-05-15,11:00:00
25268278,2025-05-15 12:00:00,2025/05/15 13:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume-les-Dames,Urbaine,PM2.5,Trafic,1,mesures fixes,7.0,7.000,2025-05-15,12:00:00


In [41]:
# check that the site don't have errors in inside
pollution_pd['nom site'].unique()

array(['Metz-Centre', 'Metz-Borny', 'Pont-à-Mousson', 'Atton',
       'Scy-Chazelles', 'Thionville-Piscine', 'Thionville-Centre',
       'HAYANGE-MARSPICH', 'Metz- Pont Grilles', 'Belleville sur Meuse',
       "Berre l'Etang", 'Martigues P. Central', 'Port de Bouc Leque',
       'Port Saint Louis', 'Istres', 'Fos Les Carabins',
       'Rognac les Brets', 'Sausset les Pins', 'Arles',
       'Chateauneuf La Mede', 'SALON', 'Marignane', 'MARSEILLE RABATAU',
       'MARSEILLE ST LOUIS', 'AIX ROY RENE', 'PLAN AUPS/STE  BAUME',
       'AIX CENTRE ECOLE ART', 'GARDANNE', 'AUBAGNE LES PASSONS',
       'VALLEE HUVEAUNE', 'MARSEILLE 5 AVENUES', 'AIX PLATANES',
       'LA SEYNE GENOUD', 'LA VALETTE/LA GARDE', 'BRIGNOLES',
       'TOULON FOCH', 'HYERES', 'ESTEREL', 'TOULON CLARET',
       'AVIGNON   MAIRIE', 'LE PONTET', 'APT', 'Avignon Semard',
       'CARPENTRAS', 'Avignon Rocade De Gaulle', 'GENNEVILLIERS',
       'PARIS 18eme', 'Place Victor Basch', 'PARIS 12eme',
       'NEUILLY-SUR-SEINE', '

In [42]:
# count the number
pollution_pd['nom site'].nunique()

527

In [43]:
# function to clean the site's name (change -, _ or . to space) & clean double space & space at the beginning & end
def clean_name(name):
    if isinstance(name, str):
        # Replace '-', '_', '.' by space
        name = re.sub(r"[-_.]", " ", name)
        # Delete the spaces at the beginning & end
        name = name.strip()
        # Clean multiple spaces
        name = re.sub(r"\s+", " ", name)
        # Put a capital letter at each beginning of each word
        name = name.title()
    return name

In [44]:
pollution_pd['nom site'] = pollution_pd['nom site'].apply(clean_name)

In [45]:
pollution_pd.head()

,Date de début,Date de fin,Organisme,Zas,nom site,type d'implantation,Polluant,type d'influence,discriminant,type d'évaluation,valeur,valeur brute,date,hours
24,2023-01-01 00:00:00,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,B,mesures fixes,5.8,5.750,2023-01-01,00:00:00
25,2023-01-01 01:00:00,2023/01/01 02:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,B,mesures fixes,6.1,6.125,2023-01-01,01:00:00
26,2023-01-01 02:00:00,2023/01/01 03:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,B,mesures fixes,4.9,4.925,2023-01-01,02:00:00
27,2023-01-01 03:00:00,2023/01/01 04:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,B,mesures fixes,5.0,4.950,2023-01-01,03:00:00
28,2023-01-01 04:00:00,2023/01/01 05:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,B,mesures fixes,4.6,4.600,2023-01-01,04:00:00


In [46]:
# we clean one
pollution_pd['nom site'].nunique()

526

In [47]:
# let's sort it to be able to differentiate it quicly
sorted_sites = sorted(pollution_pd['nom site'].unique())
sorted_sites

['A7 Salaise Ouest',
 'A7 Sud Lyonnais',
 'A7 Valence Est',
 'Abymes Rn1',
 'Agathois Piscénois',
 'Airvault Stade Laillé',
 'Aix Centre Ecole Art',
 'Aix Platanes',
 'Aix Roy Rene',
 'Ajaccio Abbatucci',
 'Ajaccio Canetto',
 'Ajaccio Confina 2',
 'Ajaccio Piataniccia',
 'Ajaccio Sposata',
 'Albertville',
 'Albi Delmas',
 'Alençon Météo France',
 'Ambes',
 'Anglet',
 'Angouleme Gambetta',
 'Annecy Rocade',
 'Annemasse',
 'Antibes Jean Moulin',
 'Appentis',
 'Apt',
 'Argenteuil',
 'Arles',
 'Arrest',
 'Atton',
 'Aubagne Les Passons',
 'Aubervilliers',
 'Aurillac Lagarde',
 'Auto A1 Saint Denis',
 'Auxerre',
 'Av Champs Elysees',
 'Avignon Mairie',
 'Avignon Rocade De Gaulle',
 'Avignon Semard',
 'Aytré',
 'Base Loisirs Poses',
 'Bassens',
 'Bastia Fango',
 'Bastia Giraud',
 'Bastia La Marana',
 'Bastia Montesoro',
 'Baume Les Dames',
 'Bd Banks',
 'Beaulieu',
 'Beaux Arts',
 'Belesta En Lauragais',
 'Belfort Centre',
 'Belfort Octroi',
 'Belleville Sur Meuse',
 'Bellevue',
 "Berre L'Eta

In [48]:
# transform in dataframe & export it
df_sites = pd.DataFrame({'nom site': sorted_sites})
df_sites

,nom site
0,A7 Salaise Ouest
1,A7 Sud Lyonnais
2,A7 Valence Est
3,Abymes Rn1
4,Agathois Piscénois
...,...
521,Zone Rurale Nord Est
522,Zone Rurale Se
523,Zone Rurale So
524,Zone Rurale Sud


In [49]:
# DONE, not need to run again
# Export in csv format to use ctrl + f (searching to see if there is similar quick observation)
#df_sites.to_csv("city.csv", index=False)

In [50]:
# what we find and should modify
pollution_pd.loc[pollution_pd['nom site'] == 'Besancon Prevoyance', 'nom site'] = 'Besancon Prévoyance'
pollution_pd.loc[pollution_pd['nom site'] == 'Caen Montalivet ', 'nom site'] = 'Caen Montalivet'
pollution_pd.loc[pollution_pd['nom site'] == 'Caiena3', 'nom site'] = 'Caiena' # the person made a mistake, instead to write ", the persons wrote 3
pollution_pd.loc[pollution_pd['nom site'] == 'Ecole De Carling(9)', 'nom site'] = 'Ecole De Carling' # the (9) is an indicator, but we don't need it here
pollution_pd.loc[pollution_pd['nom site'] == 'Gard Rhodanien2', 'nom site'] = 'Gard Rhodanien' # can't find the 2 on internet
pollution_pd.loc[pollution_pd['nom site'] == 'La Pallice3 Lr', 'nom site'] = 'La Pallice Lr' # same reasons
pollution_pd.loc[pollution_pd['nom site'] == 'S Etienne De Montluc', 'nom site'] = 'St Etienne De Montluc' # it is Saint
pollution_pd.loc[pollution_pd['nom site'] == 'Le Casset2', 'nom site'] = 'Le Casset 2'
pollution_pd.loc[pollution_pd['nom site'] == 'Schoeneck (19)', 'nom site'] = 'Schoeneck'
pollution_pd.loc[pollution_pd['nom site'] == 'Spicheren(14)', 'nom site'] = 'Spicheren'

In [51]:
# Filter to have the average value in one day
from datetime import time

pollution_pd['hours'] = pd.to_datetime(pollution_pd['hours'], format='%H:%M:%S').dt.time

In [52]:
# 2023-01-01	NO2	Saint Exupery	48
# Check the merging prb of 48h

pollution_pd[
    (pollution_pd['date'] == '2023-01-01') &
    (pollution_pd['Polluant'] == 'NO2') &
    (pollution_pd['nom site'] == 'Saint Exupery')
]

,Date de début,Date de fin,Organisme,Zas,nom site,type d'implantation,Polluant,type d'influence,discriminant,type d'évaluation,valeur,valeur brute,date,hours


In [52]:
# Check we have 24 hours for each
pollution_pd.groupby(['date', 'Polluant', "nom site"]).size().reset_index(name='total')

,date,Polluant,nom site,total
0,2023-01-01,NO2,A7 Salaise Ouest,24
1,2023-01-01,NO2,A7 Sud Lyonnais,24
2,2023-01-01,NO2,A7 Valence Est,24
3,2023-01-01,NO2,Abymes Rn1,24
4,2023-01-01,NO2,Agathois Piscénois,24
...,...,...,...,...
612517,2025-05-15,PM2.5,Wattignies,15
612518,2025-05-15,PM2.5,Zone Rurale Nord,15
612519,2025-05-15,PM2.5,Zone Rurale Se,15
612520,2025-05-15,PM2.5,Zone Rurale Sud,15


In [53]:
# create a variable to know how much of 24 hours or 15 hours we had what is the content inside it
nb_hours = pollution_pd.groupby(['date', 'Polluant', "nom site"]).size().reset_index(name='total')

In [64]:
# transform to int, because we can't sum object
nb_hours['total'] = nb_hours['total'].astype(int)

In [65]:
# Check the unique value inside the one day
nb_hours['total'].unique()
# It is not normal, that we have less than 24, but it means that we are missing somes values, 
# but it makes no sense that we exceed 24, it means we duplicate it

array([24, 23, 48, 10, 22, 16,  8, 20, 19, 21, 13, 11, 12,  9,  2, 14,  7,
        5,  4, 15, 18, 47, 17,  6,  1,  3, 46, 41, 43, 33, 38, 45, 44, 31,
       42, 34, 32, 40, 37, 30, 27, 25, 35, 39, 36, 28, 29])

In [66]:
# if the balance is fair between all the date
nb_hours.groupby('total').count()
# let's first check the date < 24
# than > 24

,date,Polluant,nom site
total,,,
1,136,136,136
2,181,181,181
3,195,195,195
4,214,214,214
5,185,185,185
6,534,534,534
7,1242,1242,1242
8,717,717,717
9,902,902,902


In [69]:
nb_hours[nb_hours["total"]>24]

,date,Polluant,nom site,total
299,2023-01-01,NO2,Saint Exupery,48
599,2023-01-01,O3,Saint Exupery,48
942,2023-01-01,PM10,Saint Exupery,48
1529,2023-01-02,NO2,Saint Exupery,48
1834,2023-01-02,O3,Saint Exupery,48
...,...,...,...,...
610427,2025-05-14,NO2,Saint Exupery,47
610706,2025-05-14,O3,Saint Exupery,47
611030,2025-05-14,PM10,Saint Exupery,41
611616,2025-05-15,NO2,Saint Exupery,29


In [68]:
nb_hours[nb_hours["total"]==1]

,date,Polluant,nom site,total
4311,2023-01-04,O3,Roanne,1
7775,2023-01-07,NO2,St De Baie Mahault,1
10108,2023-01-09,NO2,Lorient B Bissonnet,1
10446,2023-01-09,O3,Lorient B Bissonnet,1
10452,2023-01-09,O3,Manosque,1
...,...,...,...,...
602682,2025-05-07,PM10,Place De Verdun,1
602836,2025-05-07,PM2.5,Chaussée Royale,1
603316,2025-05-08,NO2,St De Baie Mahault,1
609219,2025-05-13,NO2,Robert Bourg,1


In [53]:
# Compute the average daily value of each indicator
daily_avg = pollution_pd.groupby(['date','nom site','Polluant'])['valeur'].mean().reset_index(name='valeur')
daily_avg_brut = pollution_pd.groupby(['date','nom site','Polluant'])['valeur brute'].mean().reset_index(name='valeur brute')

In [54]:
daily_avg

,date,nom site,Polluant,valeur
0,2023-01-01,A7 Salaise Ouest,NO2,12.262500
1,2023-01-01,A7 Salaise Ouest,PM10,23.079167
2,2023-01-01,A7 Salaise Ouest,PM2.5,9.008333
3,2023-01-01,A7 Sud Lyonnais,NO2,6.437500
4,2023-01-01,A7 Sud Lyonnais,PM10,22.900000
...,...,...,...,...
612517,2025-05-15,Zone Rurale Sud,PM2.5,12.006667
612518,2025-05-15,Zoodyssée Chizé,NO2,3.933333
612519,2025-05-15,Zoodyssée Chizé,O3,89.960000
612520,2025-05-15,Zoodyssée Chizé,PM10,17.366667


In [55]:
daily_avg_brut

,date,nom site,Polluant,valeur brute
0,2023-01-01,A7 Salaise Ouest,NO2,12.237500
1,2023-01-01,A7 Salaise Ouest,PM10,23.077083
2,2023-01-01,A7 Salaise Ouest,PM2.5,8.992708
3,2023-01-01,A7 Sud Lyonnais,NO2,6.417361
4,2023-01-01,A7 Sud Lyonnais,PM10,22.893750
...,...,...,...,...
612517,2025-05-15,Zone Rurale Sud,PM2.5,11.988333
612518,2025-05-15,Zoodyssée Chizé,NO2,3.925000
612519,2025-05-15,Zoodyssée Chizé,O3,89.938333
612520,2025-05-15,Zoodyssée Chizé,PM10,17.353333


In [56]:
# drop the duplicates/same columns (here, the different hours in a day)
pollution_pd = pollution_pd.drop_duplicates(subset=['date', 'Polluant', "nom site"])

In [57]:
# drop the hours columns, we have more than one columns & daily_avg or daily_avg_brut only have one, so the merging makes sense
pollution_pd = pollution_pd.drop(columns="hours")

In [58]:
# Merging the average of daily_avg & daily_avg_brut
pollution_pd = pollution_pd.merge(
    daily_avg[['nom site', 'date', 'Polluant']], 
    on=['nom site', 'date', 'Polluant'], 
    how='left'
)

pollution_pd = pollution_pd.merge(
    daily_avg_brut[['nom site', 'date', 'Polluant']], 
    on=['nom site', 'date', 'Polluant'], 
    how='left'
)

In [59]:
pollution_pd

,Date de début,Date de fin,Organisme,Zas,nom site,type d'implantation,Polluant,type d'influence,discriminant,type d'évaluation,valeur,valeur brute,date
0,2023-01-01,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,B,mesures fixes,5.8,5.750,2023-01-01
1,2023-01-01,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,O3,Fond,A,mesures fixes,39.7,39.725,2023-01-01
2,2023-01-01,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,PM10,Fond,C,mesures fixes,9.5,9.450,2023-01-01
3,2023-01-01,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,PM2.5,Fond,2,mesures fixes,7.3,7.325,2023-01-01
4,2023-01-01,2023/01/01 01:00:00,ATMO GRAND EST,ZAG METZ,Metz Borny,Urbaine,NO2,Fond,1,mesures fixes,4.8,4.750,2023-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
612517,2025-05-15,2025/05/15 01:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Vesoul Pres Caillet,Urbaine,PM10,Fond,2,mesures fixes,10.2,10.200,2025-05-15
612518,2025-05-15,2025/05/15 01:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Vesoul Pres Caillet,Urbaine,PM2.5,Fond,2,mesures fixes,7.8,7.775,2025-05-15
612519,2025-05-15,2025/05/15 01:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume Les Dames,Urbaine,NO2,Trafic,1,mesures fixes,4.7,4.700,2025-05-15
612520,2025-05-15,2025/05/15 01:00:00,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume Les Dames,Urbaine,PM10,Trafic,1,mesures fixes,9.9,9.900,2025-05-15


In [60]:
# check if the valeur & valeur brute have errors or missing value
pollution_pd.isna().sum()

Date de début              0
Date de fin                0
Organisme                  0
Zas                        0
nom site                   0
type d'implantation        0
Polluant                   0
type d'influence           0
discriminant           77091
type d'évaluation          0
valeur                     0
valeur brute               0
date                       0
dtype: int64

In [67]:
# verify that we only have 1 unique row in for a in a site from 1 indicator
nb_hours = pollution_pd.groupby(['date', 'Polluant', "nom site"]).size().reset_index(name='total').sort_values(by=['Polluant', 'total'], ascending=[True, False])
nb_hours

,date,Polluant,nom site,total
0,2023-01-01,NO2,A7 Salaise Ouest,1
1,2023-01-01,NO2,A7 Sud Lyonnais,1
2,2023-01-01,NO2,A7 Valence Est,1
3,2023-01-01,NO2,Abymes Rn1,1
4,2023-01-01,NO2,Agathois Piscénois,1
...,...,...,...,...
612517,2025-05-15,PM2.5,Wattignies,1
612518,2025-05-15,PM2.5,Zone Rurale Nord,1
612519,2025-05-15,PM2.5,Zone Rurale Se,1
612520,2025-05-15,PM2.5,Zone Rurale Sud,1


In [68]:
# if the balance is fair between all the date
nb_hours.groupby('total').count()

,date,Polluant,nom site
total,,,
1,612522,612522,612522


In [62]:
pollution_pd = pollution_pd.drop(columns="Date de début") # not need anymore
pollution_pd = pollution_pd.drop(columns="Date de fin") # not need anymore
#pollution_pd = pollution_pd.drop(columns="valeur") # ?
#pollution_pd = pollution_pd.drop(columns="valeur brute") # the approximation of value

In [63]:
pollution_pd.groupby(['discriminant'])['valeur'].mean().reset_index(name='valeur').sort_values(by='valeur', ascending=False)

,discriminant,valeur
10,A,28.172067
0,0,22.485091
1,1,20.852606
19,K,15.545224
21,O,15.331452
17,H,15.311616
13,D,15.091811
11,B,14.601277
8,8,13.962878
4,4,13.881373


In [64]:
pollution_pd = pollution_pd.drop(columns="discriminant") # we don't have the documentation, we can't guess it. We will assume it will not affected our analysis
pollution_pd = pollution_pd.drop(columns="type d'évaluation") # we will assume that the evaluation is good enough

In [65]:
pollution_pd

,Organisme,Zas,nom site,type d'implantation,Polluant,type d'influence,valeur,valeur brute,date
0,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,NO2,Fond,5.8,5.750,2023-01-01
1,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,O3,Fond,39.7,39.725,2023-01-01
2,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,PM10,Fond,9.5,9.450,2023-01-01
3,ATMO GRAND EST,ZAG METZ,Metz Centre,Urbaine,PM2.5,Fond,7.3,7.325,2023-01-01
4,ATMO GRAND EST,ZAG METZ,Metz Borny,Urbaine,NO2,Fond,4.8,4.750,2023-01-01
...,...,...,...,...,...,...,...,...,...
612517,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Vesoul Pres Caillet,Urbaine,PM10,Fond,10.2,10.200,2025-05-15
612518,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Vesoul Pres Caillet,Urbaine,PM2.5,Fond,7.8,7.775,2025-05-15
612519,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume Les Dames,Urbaine,NO2,Trafic,4.7,4.700,2025-05-15
612520,ATMO BOURGOGNE-FRANCHE-COMTE,ZR BOURGOGNE-FRANCHE-COMTE,Baume Les Dames,Urbaine,PM10,Trafic,9.9,9.900,2025-05-15


In [69]:
# Export in csv format
pollution_pd.to_csv("pollution.csv", index=False)

### Weather index

files name .csv.gz:

QUOT_departement_01_periode_1852-1949_RR-T-Vent
QUOT_departement_01_periode_1852-1949_autres-parametres
QUOT_departement_01_periode_1950-2023_RR-T-Vent
QUOT_departement_01_periode_1950-2023_autres-parametres
QUOT_departement_01_periode_2024-2025_RR-T-Vent
QUOT_departement_01_periode_2024-2025_autres-parametres
QUOT_departement_02_periode_1864-1949_RR-T-Vent
QUOT_departement_02_periode_1864-1949_autres-parametres
QUOT_departement_02_periode_1950-2023_RR-T-Vent
QUOT_departement_02_periode_1950-2023_autres-parametres
QUOT_departement_02_periode_2024-2025_RR-T-Vent
QUOT_departement_02_periode_2024-2025_autres-parametres
QUOT_departement_03_periode_1871-1949_RR-T-Vent
QUOT_departement_03_periode_1871-1949_autres-parametres
QUOT_departement_03_periode_1950-2023_RR-T-Vent
QUOT_departement_03_periode_1950-2023_autres-parametres
QUOT_departement_03_periode_2024-2025_RR-T-Vent
QUOT_departement_03_periode_2024-2025_autres-parametres
QUOT_departement_04_periode_1872-1949_RR-T-

## Merging

In [27]:
# List of all *_clean.csv files
clean_files = glob.glob("meteo_data_csv/*_clean.csv")

# Columns we expect
selected_cols = [
    "AAAAMMJJ", "NOM_USUEL", "LAT", "LON",  # Location/date
    "T_Q", "TINF_H_Q", "TSUP_H_Q",         # Temperature
    "PRELIQ_Q", "PE_Q", "RUNC_Q", "DRAINC_Q",  # Water
    "SWI_Q", "HU_Q", "Q_Q",                # Soil/atmo
    "ETP_Q", "EVAP_Q", "FF_Q"              # Evaporation/wind
]

# Desired schema (can be customized if needed)
expected_schema = {
    "AAAAMMJJ": pl.Int64,
    "NOM_USUEL": pl.Utf8,
    "LAT": pl.Float64,
    "LON": pl.Float64,
    "T_Q": pl.Float64,
    "TINF_H_Q": pl.Float64,
    "TSUP_H_Q": pl.Float64,
    "PRELIQ_Q": pl.Float64,
    "PE_Q": pl.Float64,
    "RUNC_Q": pl.Float64,
    "DRAINC_Q": pl.Float64,
    "SWI_Q": pl.Float64,
    "HU_Q": pl.Float64,
    "Q_Q": pl.Float64,
    "ETP_Q": pl.Float64,
    "EVAP_Q": pl.Float64,
    "FF_Q": pl.Float64
}
clean_files

['meteo_data_csv\\Q_01_1852-1949_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_01_latest-2024-2025_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_01_previous-1950-2023_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_02_1864-1949_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_02_latest-2024-2025_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_02_previous-1950-2023_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_03_1871-1949_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_03_latest-2024-2025_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_03_previous-1950-2023_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_04_1872-1949_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_04_latest-2024-2025_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_04_previous-1950-2023_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_05_1877-1949_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_05_latest-2024-2025_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_05_previous-1950-2023_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_06_1877-1949_RR-T-Vent_clean.csv',
 'meteo_data_csv\\Q_06_latest-2024-2025_

In [20]:
# Define the common schema manually
schema = {
    "NUM_POSTE": pl.Int64,
    "NOM_USUEL": pl.String,
    "LAT": pl.Float64,
    "LON": pl.Float64,
    "ALTI": pl.Int64,
    "AAAAMMJJ": pl.Int64,
    "RR": pl.Float64,
    "QRR": pl.Int64,
    "TN": pl.Float64,
    "QTN": pl.Int64,
    "HTN": pl.Int64,
    "QHTN": pl.Int64,
    "TX": pl.Float64,
    "QTX": pl.Int64,
    "HTX": pl.Int64,
    "QHTX": pl.Int64,
    "TM": pl.Float64,
    "QTM": pl.Int64,
    "TNTXM": pl.Float64,
    "QTNTXM": pl.Int64,
    "TAMPLI": pl.Float64,
    "QTAMPLI": pl.Int64,
    "TNSOL": pl.Float64,
    "QTNSOL": pl.Int64,
    "TN50": pl.Float64,
    "QTN50": pl.Int64,
    "DG": pl.Int64,
    "QDG": pl.Int64,
    "FFM": pl.Float64,
    "QFFM": pl.Int64,
    "FF2M": pl.Float64,
    "QFF2M": pl.Int64,
    "FXY": pl.Float64,
    "QFXY": pl.Int64,
    "DXY": pl.Int64,
    "QDXY": pl.Int64,
    "HXY": pl.Int64,
    "QHXY": pl.Int64,
    "FXI": pl.Float64,
    "QFXI": pl.Int64,
    "DXI": pl.Int64,
    "QDXI": pl.Int64,
    "HXI": pl.Int64,
    "QHXI": pl.Int64,
    "FXI2": pl.Float64,
    "QFXI2": pl.Int64,
    "DXI2": pl.Int64,
    "QDXI2": pl.Int64,
    "HXI2": pl.Int64,
    "QHXI2": pl.Int64,
    "FXI3S": pl.Float64,
    "QFXI3S": pl.Int64,
    "DXI3S": pl.Int64,
    "QDXI3S": pl.Int64,
    "HXI3S": pl.Int64,
    "QHXI3S": pl.Int64,
    "DRR": pl.Float64,
    "QDRR": pl.Int64
}

In [25]:
# Search for all filtered CSV files
clean_files = glob.glob("meteo_data_csv/*_clean.csv") #_filtered

# Prepare lazy loading of each file
lazy_frames = []
for f in clean_files:
    try:
        schema = pl.scan_csv(f, separator=",").schema
        print(f"{f} → {list(schema.keys())}")
    except Exception as e:
        print(f"Error reading {f}: {e}")


# merge all the files in lazy mode
df_lazy = pl.concat(lazy_frames)

# Collect
df_filtered = df_lazy.collect()

# Save
df_filtered.write_csv("weather.csv")
print(f"Merging of {len(clean_files)} files → total : {df.shape[0]} lines")

C:\Users\eric-\AppData\Local\Temp\ipykernel_23944\1346953791.py:8: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = pl.scan_csv(f, separator=",").schema


meteo_data_csv\Q_01_1852-1949_RR-T-Vent_clean.csv → ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR', 'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM', 'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50', 'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY', 'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI', 'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S', 'QFXI3S', 'DXI3S', 'QDXI3S', 'HXI3S', 'QHXI3S', 'DRR', 'QDRR']
meteo_data_csv\Q_01_latest-2024-2025_RR-T-Vent_clean.csv → ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR', 'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM', 'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50', 'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY', 'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI', 'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S'

ValueError: cannot concat empty list

# Cleaning

In [15]:
df_filtered

AAAAMMJJ,NOM_USUEL,LAT,LON
i64,str,f64,f64
null,"""AMBERIEU""",45.955,5.348333
null,"""AMBERIEU""",45.955,5.348333
null,"""AMBERIEU""",45.955,5.348333
null,"""AMBERIEU""",45.955,5.348333
null,"""AMBERIEU""",45.955,5.348333
…,…,…,…
20231227,"""WY-DIT""",49.108167,1.830667
20231228,"""WY-DIT""",49.108167,1.830667
20231229,"""WY-DIT""",49.108167,1.830667


In [54]:
null_counts = df.null_count()
null_counts.select(null_counts.columns[5:])

AAAAMMJJ,RR,QRR,TN,QTN,HTN,QHTN,TX,QTX,HTX,QHTX,TM,QTM,TNTXM,QTNTXM,TAMPLI,QTAMPLI,TNSOL,QTNSOL,TN50,QTN50,DG,QDG,FFM,QFFM,FF2M,QFF2M,FXY,QFXY,DXY,QDXY,HXY,QHXY,FXI,QFXI,DXI,QDXI,HXI,QHXI,FXI2,QFXI2,DXI2,QDXI2,HXI2,QHXI2,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
314901,327,290,146779,146700,198772,198561,146697,146673,198728,198640,172882,172880,146858,146844,146858,146844,213847,213826,297706,297704,211985,211966,224285,224268,321845,321842,239643,239593,239965,239912,251621,251555,223113,223092,232508,232485,250720,250679,321845,321842,321845,321842,321845,321842,304490,304458,321845,321810,304490,304458,286982,286502


In [58]:
df.describe()  #  info

statistic,NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJ,RR,QRR,TN,QTN,HTN,QHTN,TX,QTX,HTX,QHTX,TM,QTM,TNTXM,QTNTXM,TAMPLI,QTAMPLI,TNSOL,QTNSOL,TN50,QTN50,DG,QDG,FFM,QFFM,FF2M,QFF2M,FXY,QFXY,DXY,QDXY,HXY,QHXY,FXI,QFXI,DXI,QDXI,HXI,QHXI,FXI2,QFXI2,DXI2,QDXI2,HXI2,QHXI2,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""",321845.0,"""321845""",321845.0,321845.0,321845.0,6944.0,321518.0,321555.0,175066.0,175145.0,123073.0,123284.0,175148.0,175172.0,123117.0,123205.0,148963.0,148965.0,174987.0,175001.0,174987.0,175001.0,107998.0,108019.0,"""24139""","""24141""",109860.0,109879.0,"""97560""","""97577""","""0""","""3""","""82202""","""82252""","""81880""","""81933""","""70224""","""70290""","""98732""","""98753""","""89337""","""89360""","""71125""","""71166""","""0""","""3""","""0""","""3""","""0""","""3""","""17355""","""17387""","""0""","""35""","""17355""","""17387""","""34863""","""35343"""
"""null_count""",0.0,"""0""",0.0,0.0,0.0,314901.0,327.0,290.0,146779.0,146700.0,198772.0,198561.0,146697.0,146673.0,198728.0,198640.0,172882.0,172880.0,146858.0,146844.0,146858.0,146844.0,213847.0,213826.0,"""297706""","""297704""",211985.0,211966.0,"""224285""","""224268""","""321845""","""321842""","""239643""","""239593""","""239965""","""239912""","""251621""","""251555""","""223113""","""223092""","""232508""","""232485""","""250720""","""250679""","""321845""","""321842""","""321845""","""321842""","""321845""","""321842""","""304490""","""304458""","""321845""","""321810""","""304490""","""304458""","""286982""","""286502"""
"""mean""",9.5343e7,null,49.062644,2.157271,81.263385,2.0220e7,1.824989,1.066231,7.202315,1.000851,732.867705,4.568022,15.724007,1.000742,1352.843994,4.638878,11.231917,1.197624,11.484409,1.000131,8.516091,1.000131,6.059145,1.585453,null,null,69.130475,5.432212,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",211964.076336,null,0.061635,0.252649,34.62449,8285.372312,4.019784,0.725025,5.764282,0.083422,632.640823,3.976622,7.699561,0.079024,274.069092,3.983682,6.446108,1.241775,6.458398,0.033209,4.276773,0.033209,5.988336,2.083484,null,null,242.870486,3.976609,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""",9.5018002e7,"""ARGENTEUIL""",48.961667,1.668333,18.0,2.0201231e7,0.0,0.0,-18.2,0.0,0.0,1.0,-11.1,0.0,0.0,1.0,-13.8,0.0,-13.3,0.0,0.1,0.0,-22.6,1.0,"""-0.1""","""1""",0.0,0.0,"""0.0""","""1""",null,"""1""","""0.0""","""1""","""0""","""1""","""0""","""0""","""0.0""","""0""","""0""","""0""","""0""","""0""",null,"""1""",null,"""1""",null,"""1""","""10.0""","""0""",null,"""1""","""0""","""1""","""0""","""0"""
"""25%""",9.5119001e7,null,49.015167,2.0285,50.0,2.0210904e7,0.0,1.0,3.0,1.0,345.0,1.0,10.0,1.0,1255.0,1.0,6.6,1.0,6.8,1.0,5.2,1.0,1.7,1.0,null,null,0.0,1.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",9.5306001e7,null,49.051667,2.154333,75.0,2.0220601e7,0.0,1.0,7.5,1.0,510.0,1.0,15.6,1.0,1405.0,1.0,11.3,1.0,11.5,1.0,7.9,1.0,6.3,1.0,null,null,0.0,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",9.5527001e7,null,49.112167,2.406667,108.0,2.0230317e7,1.8,1.0,11.6,1.0,723.0,9.0,21.5,1.0,1501.0,9.0,16.1,1.0,16.5,1.0,11.4,1.0,10.6,1.0,null,null,0.0,9.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""",9.5690001e7,"""WY-DIT""",49.194,

In [49]:
df.select(df.columns[5:])

AAAAMMJJ,RR,QRR,TN,QTN,HTN,QHTN,TX,QTX,HTX,QHTX,TM,QTM,TNTXM,QTNTXM,TAMPLI,QTAMPLI,TNSOL,QTNSOL,TN50,QTN50,DG,QDG,FFM,QFFM,FF2M,QFF2M,FXY,QFXY,DXY,QDXY,HXY,QHXY,FXI,QFXI,DXI,QDXI,HXI,QHXI,FXI2,QFXI2,DXI2,QDXI2,HXI2,QHXI2,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR
i64,f64,i64,f64,i64,i64,i64,f64,i64,i64,i64,f64,i64,f64,i64,f64,i64,f64,i64,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
19940201,0.0,1,3.7,1,245,1,9.4,1,2115,1,6.9,1,6.6,1,5.7,1,2.7,1,null,null,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
19940202,11.2,1,4.8,1,1715,1,8.4,1,545,1,5.7,1,6.6,1,3.6,1,4.6,1,null,null,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
19940203,5.6,1,4.9,1,1815,1,10.5,1,1415,1,8.7,1,7.7,1,5.6,1,4.9,1,null,null,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
19940204,0.6,1,5.0,1,745,1,10.1,1,1145,1,7.7,1,7.6,1,5.1,1,3.1,1,null,null,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
19940205,0.0,1,5.5,1,115,1,9.6,1,1315,1,7.3,1,7.6,1,4.1,1,4.5,1,null,null,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,0.2,1,8.1,1,822,9,10.2,1,29,9,9.0,1,9.2,1,2.1,1,null,null,null,null,0,9,"""6.3""","""1""",null,null,"""12.1""","""1""","""210""","""1""","""1829""","""9""","""17.7""","""1""","""220""","""1""","""1842""","""9""",null,null,null,null,null,null,"""16.8""","""1""",null,null,"""1842""","""9""",null,null
null,0.0,1,7.9,1,648,9,11.7,1,1137,9,9.5,1,9.8,1,3.8,1,null,null,null,null,0,9,"""6.1""","""1""",null,null,"""9.1""","""1""","""230""","""1""","""1410""","""9""","""13.9""","""1""","""250""","""1""","""1213""","""9""",null,null,null,null,null,null,"""13.4""","""1""",null,null,"""1213""","""9""",null,null
null,1.0,1,8.0,1,2318,9,11.3,1,1207,9,9.6,1,9.7,1,3.3,1,null,null,null,null,0,9,"""7.0""","""1""",null,null,"""10.3""","""1""","""230""","""1""","""1217""","""9""","""15.4""","""1""","""220""","""1""","""1209""","""9""",null,null,null,null,null,null,"""14.9""","""1""",null,null,"""1209""","""9""",null,null


In [59]:
df = df.filter(pl.col("AAAAMMJJ") > 20201232)

In [60]:
df

NUM_POSTE,NOM_USUEL,LAT,LON,ALTI,AAAAMMJJ,RR,QRR,TN,QTN,HTN,QHTN,TX,QTX,HTX,QHTX,TM,QTM,TNTXM,QTNTXM,TAMPLI,QTAMPLI,TNSOL,QTNSOL,TN50,QTN50,DG,QDG,FFM,QFFM,FF2M,QFF2M,FXY,QFXY,DXY,QDXY,HXY,QHXY,FXI,QFXI,DXI,QDXI,HXI,QHXI,FXI2,QFXI2,DXI2,QDXI2,HXI2,QHXI2,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR
i64,str,f64,f64,i64,i64,f64,i64,f64,i64,i64,i64,f64,i64,i64,i64,f64,i64,f64,i64,f64,i64,f64,i64,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
95078001,"""PONTOISE - AERO""",49.090333,2.0285,87,20210101,0.0,1,-2.2,1,16,9,4.0,1,1320,9,0.0,1,0.9,1,6.2,1,-4.2,9,null,null,945,9,"""1.5""","""1""",null,null,"""3.9""","""1""","""10""","""1""","""1245""","""9""","""5.4""","""1""","""10""","""1""","""1240""","""9""",null,null,null,null,null,null,"""5.3""","""1""",null,null,"""1240""","""9""","""0""","""9"""
95078001,"""PONTOISE - AERO""",49.090333,2.0285,87,20210102,0.0,1,-4.5,1,721,9,1.9,1,450,9,-0.9,1,-1.3,1,6.4,1,-6.0,9,null,null,744,9,"""0.8""","""1""",null,null,"""2.3""","""1""","""300""","""1""","""1603""","""9""","""2.8""","""1""","""300""","""1""","""1601""","""9""",null,null,null,null,null,null,"""2.8""","""1""",null,null,"""1610""","""9""","""0""","""9"""
95078001,"""PONTOISE - AERO""",49.090333,2.0285,87,20210103,0.0,1,0.7,1,1853,9,3.4,1,1259,9,2.2,1,2.1,1,2.7,1,0.7,9,null,null,0,9,"""2.6""","""1""",null,null,"""5.3""","""1""","""40""","""9""","""2233""","""9""","""7.7""","""1""","""40""","""9""","""2318""","""9""",null,null,null,null,null,null,"""7.5""","""1""",null,null,"""2014""","""9""","""0""","""9"""
95078001,"""PONTOISE - AERO""",49.090333,2.0285,87,20210104,0.0,1,1.9,1,356,9,3.2,1,1208,9,2.3,1,2.6,1,1.3,1,1.7,9,null,null,0,9,"""4.2""","""1""",null,null,"""6.1""","""1""","""30""","""1""","""314""","""9""","""9.3""","""1""","""30""","""1""","""308""","""9""",null,null,null,null,null,null,"""9.1""","""1""",null,null,"""310""","""9""","""88""","""9"""
95078001,"""PONTOISE - AERO""",49.090333,2.0285,87,20210105,0.6,1,1.0,1,936,9,2.6,1,1521,9,2.2,1,1.8,1,1.6,1,0.7,9,null,null,0,9,"""3.6""","""1""",null,null,"""5.2""","""1""","""40""","""9""","""2058""","""9""","""7.3""","""1""","""40""","""9""","""2050""","""9""",null,null,null,null,null,null,"""7.1""","""1""",null,null,"""2050""","""9""","""11""","""9"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
95690001,"""WY-DIT""",49.108167,1.830667,126,20231227,0.2,1,8.1,1,822,9,10.2,1,29,9,9.0,1,9.2,1,2.1,1,null,null,null,null,0,9,"""6.3""","""1""",null,null,"""12.1""","""1""","""210""","""1""","""1829""","""9""","""17.7""","""1""","""220""","""1""","""1842""","""9""",null,null,null,null,null,null,"""16.8""","""1""",null,null,"""1842""","""9""",null,null
95690001,"""WY-DIT""",49.108167,1.830667,126,20231228,0.0,1,7.9,1,648,9,11.7,1,1137,9,9.5,1,9.8,1,3.8,1,null,null,null,null,0,9,"""6.1""","""1""",null,null,"""9.1""","""1""","""230""","""1""","""1410""","""9""","""13.9""","""1""","""250""","""1""","""1213""","""9""",null,null,null,null,null,null,"""13.4""","""1""",null,null,"""1213""","""9""",null,null
95690001,"""WY-DIT""",49.108167,1.830667,126,20231229,1.0,1,8.0,1,2318,9,11.3,1,1207,9,9.6,1,9.7,1,3.3,1,null,null,null,null,0,9,"""7.0""","""1""",null,null,"""10.3""","""1""","""230""","""1""","""1217""","""9""","""15.4""","""1""","""220""","""1""","""1209""","""9""",null,null,null,null,null,null,"""14.9""","""1""",null,null,"""1209""","""9""",null,null


In [61]:
df_pd = df.to_pandas()

In [65]:
df_pd.groupby('NOM_USUEL').count()

,NUM_POSTE,LAT,LON,ALTI,AAAAMMJJ,RR,QRR,TN,QTN,HTN,...,HXI2,QHXI2,FXI3S,QFXI3S,DXI3S,QDXI3S,HXI3S,QHXI3S,DRR,QDRR
NOM_USUEL,,,,,,,,,,,,,,,,,,,,,
EAUBONNE,365,365,365,365,365,365,365,0,0,0,...,0,0,0,0,0,0,0,0,0,0
LE BOURGET,1095,1095,1095,1095,1095,1095,1095,1095,1095,1095,...,0,0,1086,1086,0,0,1086,1086,1076,1080
LE PLESSIS GASSOT,1095,1095,1095,1095,1095,1095,1095,1095,1095,1086,...,0,0,0,0,0,0,0,0,1080,1080
PONTOISE - AERO,1095,1095,1095,1095,1095,1095,1095,1095,1095,1095,...,0,0,1093,1093,0,0,1093,1093,1011,1024
ROISSY,1095,1095,1095,1095,1095,1095,1095,1095,1095,1095,...,0,0,1075,1075,0,0,1075,1075,1088,1089
ST WITZ,1095,1095,1095,1095,1095,1095,1095,1095,1095,1093,...,0,0,0,0,0,0,0,0,0,0
WY-DIT,1095,1095,1095,1095,1095,1095,1095,1095,1095,1090,...,0,0,1029,1037,0,8,1029,1037,0,0


In [ ]:
df.head()

# Bibliography :